In [15]:
import pandas as pd
import numpy as np 
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split as tts
from sklearn.preprocessing import OneHotEncoder as ohe
from sklearn.preprocessing import OrdinalEncoder as oe
from sklearn.preprocessing import MinMaxScaler as mms
from sklearn.tree import DecisionTreeClassifier as dtc
from sklearn.impute import SimpleImputer as si

In [2]:
df = pd.read_csv('train.csv')

In [3]:
df.head()

,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,NaN,S
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C
2,3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,C123,S
4,5,0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,NaN,S


In [4]:
df.isnull().sum()

PassengerId      0
Survived         0
Pclass           0
Name             0
Sex              0
Age            177
SibSp            0
Parch            0
Ticket           0
Fare             0
Cabin          687
Embarked         2
dtype: int64

## Without pipelines

In [8]:
df.drop(columns = ['PassengerId', 'Name', 'Ticket', 'Cabin'], inplace=True)

In [9]:
df.head()

,Survived,Pclass,Sex,Age,SibSp,Parch,Fare,Embarked
0,0,3,male,22.0,1,0,7.2500,S
1,1,1,female,38.0,1,0,71.2833,C
2,1,3,female,26.0,0,0,7.9250,S
3,1,1,female,35.0,1,0,53.1000,S
4,0,3,male,35.0,0,0,8.0500,S


In [11]:
X_train, X_test, y_train, y_test = tts(df.drop(columns = ['Survived']),
                                      df['Survived'],
                                      test_size = 0.2,
                                      random_state = 0)

In [12]:
X_train.shape

(712, 7)

In [13]:
df.shape

(891, 8)

## Imputation

In [17]:
si_age = si()
si_embarked = si(strategy='most_frequent')

In [18]:
X_train_age = si_age.fit_transform(X_train[['Age']])

In [19]:
X_train_embarked = si_embarked.fit_transform(X_train[['Embarked']])

In [20]:
X_test_age = si_age.transform(X_test[['Age']])

In [40]:
X_test_age.shape

(179, 1)

In [41]:
X_test.shape

(179, 7)

In [21]:
X_test_embarked = si_embarked.transform(X_test[['Embarked']])

## One Hot Encoding on Sex and Embarked

In [37]:
df.head(2)

,Survived,Pclass,Sex,Age,SibSp,Parch,Fare,Embarked
0,0,3,male,22.0,1,0,7.2500,S
1,1,1,female,38.0,1,0,71.2833,C


In [23]:
ohe_sex = ohe(sparse=False, handle_unknown = 'ignore')

In [24]:
ohe_embarked = ohe(sparse=False, handle_unknown = 'ignore')

In [38]:
ohe_sex

OneHotEncoder(handle_unknown='ignore', sparse=False, sparse_output=False)

In [27]:
X_train_sex = ohe_sex.fit_transform(X_train[['Sex']])

C:\Users\dell\anaconda3\Lib\site-packages\sklearn\preprocessing\_encoders.py:972: FutureWarning: `sparse` was renamed to `sparse_output` in version 1.2 and will be removed in 1.4. `sparse_output` is ignored unless you leave `sparse` to its default value.
  warnings.warn(


In [39]:
X_train_sex

array([[1., 0.],
       [0., 1.],
       [0., 1.],
       ...,
       [0., 1.],
       [1., 0.],
       [0., 1.]])

In [28]:
X_train_embarked = ohe_embarked.fit_transform(X_train_embarked)

C:\Users\dell\anaconda3\Lib\site-packages\sklearn\preprocessing\_encoders.py:972: FutureWarning: `sparse` was renamed to `sparse_output` in version 1.2 and will be removed in 1.4. `sparse_output` is ignored unless you leave `sparse` to its default value.
  warnings.warn(


In [29]:
X_test_sex = ohe_sex.transform(X_test[['Sex']])

In [30]:
X_test_embarked = ohe_embarked.transform(X_test_embarked)

In [31]:
X_train.head(2)

,Pclass,Sex,Age,SibSp,Parch,Fare,Embarked
140,3,female,NaN,0,2,15.2458,C
439,2,male,31.0,0,0,10.5000,S


In [32]:
X_train_rem = X_train.drop(columns = ['Sex', 'Age', 'Embarked'])

In [ ]:
X_

In [42]:
X_test_rem = X_test.drop(columns = ['Sex', 'Age', 'Embarked'])

In [33]:
X_train_rem.head(3)

,Pclass,SibSp,Parch,Fare
140,3,0,2,15.2458
439,2,0,0,10.5000
817,2,1,1,37.0042


In [34]:
X_train_transformed = np.concatenate((X_train_rem, X_train_age,
                                     X_train_sex, X_train_embarked),
                                    axis =1)

In [43]:
X_test_transformed = np.concatenate((X_test_rem, X_test_age,
                                     X_test_sex, X_test_embarked),
                                    axis =1)

In [44]:
X_test_transformed.shape

(179, 10)

In [45]:
X_test_transformed

array([[3., 0., 0., ..., 1., 0., 0.],
       [3., 0., 0., ..., 0., 0., 1.],
       [3., 4., 1., ..., 0., 1., 0.],
       ...,
       [1., 1., 0., ..., 1., 0., 0.],
       [3., 0., 0., ..., 0., 0., 1.],
       [3., 0., 0., ..., 0., 0., 1.]])

## Machine Learning Algorithms

In [46]:
clf = dtc()

In [47]:
clf.fit(X_train_transformed, y_train)

DecisionTreeClassifier()

In [54]:
y_pred = clf.predict(X_test_transformed)
y_pred

array([0, 0, 0, 1, 0, 0, 1, 1, 1, 1, 1, 0, 0, 1, 1, 0, 0, 0, 0, 1, 0, 0,
       0, 0, 0, 1, 0, 1, 1, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
       1, 0, 0, 1, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 1, 1, 1, 0, 1, 0,
       1, 0, 0, 1, 1, 0, 1, 0, 0, 0, 1, 1, 0, 1, 0, 1, 1, 0, 0, 1, 1, 1,
       1, 0, 0, 0, 1, 1, 0, 0, 1, 0, 0, 0, 0, 0, 0, 1, 0, 0, 1, 0, 0, 1,
       0, 1, 1, 0, 1, 1, 0, 1, 0, 1, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0,
       0, 0, 0, 0, 0, 1, 0, 0, 0, 1, 0, 1, 1, 0, 0, 0, 1, 1, 0, 0, 1, 1,
       0, 0, 0, 0, 1, 1, 0, 0, 1, 0, 0, 0, 0, 1, 0, 0, 0, 1, 0, 0, 1, 0,
       1, 0, 1], dtype=int64)

In [59]:
from sklearn.metrics import accuracy_score

In [60]:
accuracy_score(y_test, y_pred)*100

75.41899441340783

In [61]:
import pickle

In [56]:
pickle.dump(ohe_sex, open('models/ohe_sex.pkl','wb'))

In [57]:
pickle.dump(ohe_embarked, open('models/ohe_embarked.pkl','wb'))

In [58]:
pickle.dump(clf, open('models/clf.pkl','wb'))

## Hello this is the cell i am using

## Hello this is markdown

# Hello this is another markdown

In [ ]:
import matplotlib.pyplot 

In [63]:
import numpy as np

In [ ]:
import pandas as pd

In [66]:
from sklearn.linear_model import LinearRegression

In [67]:
lr = LinearRegression()

In [68]:
from sklearn.linear_model import LogisticRegression

In [69]:
lor = LogisticRegression()

In [70]:
!cd

C:\Users\dell\100 Days of Machine Learning


In [72]:
!git init

Initialized empty Git repository in C:/Users/dell/100 Days of Machine Learning/.git/


In [73]:
import warnings
warnings.filterwarnings('ignore')

In [74]:
Out

{3:    PassengerId  Survived  Pclass  \
 0            1         0       3   
 1            2         1       1   
 2            3         1       3   
 3            4         1       1   
 4            5         0       3   
 
                                                 Name     Sex   Age  SibSp  \
 0                            Braund, Mr. Owen Harris    male  22.0      1   
 1  Cumings, Mrs. John Bradley (Florence Briggs Th...  female  38.0      1   
 2                             Heikkinen, Miss. Laina  female  26.0      0   
 3       Futrelle, Mrs. Jacques Heath (Lily May Peel)  female  35.0      1   
 4                           Allen, Mr. William Henry    male  35.0      0   
 
    Parch            Ticket     Fare Cabin Embarked  
 0      0         A/5 21171   7.2500   NaN        S  
 1      0          PC 17599  71.2833   C85        C  
 2      0  STON/O2. 3101282   7.9250   NaN        S  
 3      0            113803  53.1000  C123        S  
 4      0            373450   8.0

In [77]:
Out[9]

,Survived,Pclass,Sex,Age,SibSp,Parch,Fare,Embarked
0,0,3,male,22.0,1,0,7.2500,S
1,1,1,female,38.0,1,0,71.2833,C
2,1,3,female,26.0,0,0,7.9250,S
3,1,1,female,35.0,1,0,53.1000,S
4,0,3,male,35.0,0,0,8.0500,S
